# Scraping The Crag

_url_: https://www.thecrag.com/

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import numpy as np

## Austria

### Routes

In [2]:
LOGIN_URL = "https://www.thecrag.com/CIDS/cgi-bin/cids.cgi"
BASE_URL = "https://www.thecrag.com"
ROUTES_URL_TEMPLATE = "https://www.thecrag.com/en/climbing/austria/routes/with-grade/AU:1:39/with-gear-style/boulder+trad+sport+top-rope/in-setting/natural/?sortby=at,desc&page={}"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

In [ ]:
def login(email, password):
    session = requests.Session()
    resp = session.get(LOGIN_URL, headers=HEADERS)
    soup = BeautifulSoup(resp.text, "html.parser")
    
    csrf_input = soup.find("input", {"name": "D:CSRF"})
    if not csrf_input:
        raise Exception("Could not find CSRF token.")
    
    csrf_token = csrf_input.get("value")

    payload = {
        "D:Login": email,
        "D:Password": password,
        "State:Login": "Anmelden",
        "D:LoginTarget": "/settings/profile?&__tccp=0_",
        "C:Portal": "Default Licensee",
        "C:ResetContinueURL": "1",
        "D:CSRF": csrf_token
    }

    login_response = session.post(LOGIN_URL, data=payload, headers=HEADERS)
    if "Logout" not in login_response.text:
        raise Exception("Login failed.")
    
    print("Logged in successfully.")
    return session


In [ ]:
def parse_grade(raw_grade):
    if not raw_grade:
        return None, None

    patterns = [
        (r"\{(\w+)\}\s*([A-Za-z0-9.+\-]+)", True),   # {FB} 7A
        (r"(\w+):([A-Za-z0-9.+\-]+)", False),        # UIAA:6+
    ]

    for pattern, swap in patterns:
        match = re.search(pattern, raw_grade)
        if match:
            system, grade = match.group(1), match.group(2)
            return grade.strip(), system.strip()

    return raw_grade.strip(), "Unknown"

def scrape_routes(session, start_page, max_pages=50):
    all_data = []

    for page in range(start_page, start_page + max_pages):
        url = ROUTES_URL_TEMPLATE.format(page)

        if (page%5==0):
            print(f"Scraping page {page}...")
        
        resp = session.get(url, headers=HEADERS)
        soup = BeautifulSoup(resp.text, "html.parser")

        route_rows = soup.find_all("tr", class_="actionable")
        if not route_rows:
            print("No more routes found. Ending scrape.")
            break

        # Find current "group trail" info for region parsing
        group_trails = soup.find_all("tr", {"class": "group"})
        current_region = ""
        orientation = ""

        group_idx = 0

        for row in route_rows:
            # ID
            route_id = row.get("data-nid")
            name = row.get("data-nodename")

            # Grade
            grade_span = row.find("span", class_="pull-right")
            grade = grade_span.text.strip() if grade_span else None

            # Gear Style
            gear_style = None
            for span in row.find_all("span"):
                if span.has_attr("class") and "tags" in span["class"]:
                    gear_style = span.text.strip() if span else np.nan
                    break 

            # Get crag and region from title
            route_name_td = row.find("td", class_="rt_name")
            if route_name_td and route_name_td.find("a"):
                title = route_name_td.find("a").get("title", "")
                crumbs = [crumb.strip() for crumb in title.split("›") if crumb.strip()]
                country = "Austria"
                crag = crumbs[-1] if crumbs else None
                region = crumbs[-2] if len(crumbs) >= 2 else None
                orientation = crumbs[3] if len(crumbs) >= 4 else None  # e.g., "Ost", "West"
            else:
                country = "Austria"
                crag = region = orientation = None

            # Grade parsing
            grade_clean, grade_system = parse_grade(grade)

            all_data.append({
                "id": route_id,
                "name": name,
                "gear_style": gear_style,
                "country": country,
                "orientation": orientation,
                "region": region,
                "crag": crag,
                "grade": grade,
                "grade_clean": grade_clean,
                "grade_system": grade_system
            })

        time.sleep(1)  # to be polite to the server

    return pd.DataFrame(all_data)


In [ ]:
email = "clarapic"
password = "CraghJhCjFl69<"
sp = 1

session = login(email, password)
for i in range(1, 6):
    df = scrape_routes(session, start_page=sp, max_pages=50)
    df.to_csv("data/theCrag/routes_austria_"+ str(i) + ".csv", index=False)
    sp =+ 50
 

Exception: Login failed.

In [ ]:
df1 = pd.read_csv("data/theCrag/routes_austria_1.csv")
df2 = pd.read_csv("data/theCrag/routes_austria_2.csv")
df3 = pd.read_csv("data/theCrag/routes_austria_3.csv")
df4 = pd.read_csv("data/theCrag/routes_austria_4.csv")
df5 = pd.read_csv("data/theCrag/routes_austria_5.csv")

df_list = [df1, df2, df3, df4, df5]
df_austria_routes = pd.concat(df_list)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            1500 non-null   object
 1   name          1500 non-null   object
 2   gear_style    1470 non-null   object
 3   country       1500 non-null   object
 4   orientation   1500 non-null   object
 5   region        1500 non-null   object
 6   crag          1500 non-null   object
 7   grade         1500 non-null   object
 8   grade_clean   1500 non-null   object
 9   grade_system  1500 non-null   object
dtypes: object(10)
memory usage: 117.3+ KB


None

,id,name,gear_style,country,orientation,region,crag,grade,grade_clean,grade_system
0,4650154410,New Wave Hooker,None,Austria,Ost,Wildon,Höhle,{FB} 7A,7A,FB
1,4650154323,Thekentraverse,None,Austria,Ost,Wildon,Höhle,{FB} 7A,7A,FB
2,4650154236,Linke Traverse,None,Austria,Ost,Wildon,Höhle,{FB} 7A,7A,FB
3,4650154149,Helicopter-Dyno,None,Austria,Ost,Wildon,Höhle,{FB} 7A+,7A+,FB
4,4650154062,Techno macht's möglich,Boulder,Austria,Ost,Wildon,Höhle,{FB} 7B,7B,FB
5,4650153975,Techno macht's möglich (extension),Boulder,Austria,Ost,Wildon,Höhle,{FB} 7C,7C,FB
6,4650153888,Rechter Boulder,Boulder,Austria,Ost,Wildon,Höhle,{FB} 7C,7C,FB
7,4650153801,Black Betty,Boulder,Austria,Ost,Wildon,Höhle,{FB} 8A,8A,FB
8,4586978976,Loch,Sport,Austria,Ost,Randgebirge östl. d. Mur,Klettergarten Markt Neuhodis,UIAA:7,7,UIAA
9,4586978895,Knopp vorbei,Sport,Austria,Ost,Randgebirge östl. d. Mur,Klettergarten Markt Neuhodis,UIAA:6+,6+,UIAA


In [ ]:
display(df_austria_routes.info())
display(df_austria_routes.head(10))
display(df_austria_routes.tail(10))

### Ascents